In [ ]:
%pip install pandas numpy seaborn matplotlib scikit-learn

# Project: ServiceNow Incident Risk Score Prediction

## 1. Business Objective Summary

**Goal:** Predict a **"Risk Score" (0–100)** for open ServiceNow incidents to identify potential escalations *before* they happen.

**Problem:** Support managers cannot manually review thousands of tickets. They need a "weighted" list of which tickets are most likely to result in customer dissatisfaction.

**Success Metric:** A high **R-squared ($R^2$)** value, indicating the model accurately understands which factors (like "bounces" or "aging") drive escalation.

## 2. The Python Implementation

### Step 1: Import Libraries and Create Synthetic Data
We start by creating a dataset that mimics a ServiceNow incident table export.

**Note:** Real-world data is never perfect. We will simulate some "messy" data (missing values) to show how to handle it.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Setting seed for reproducibility
np.random.seed(42)

# Creating a dataset that looks like a ServiceNow Export
data = {
    'incident_id': [f'INC00{i}' for i in range(100)],
    'priority': np.random.randint(1, 5, 100),           # 1 (High) to 4 (Low)
    'reassignment_count': np.random.randint(0, 10, 100),# How many times it "bounced"
    'age_days': np.random.randint(1, 30, 100),          # Days open
    'sentiment_score': np.random.uniform(0, 1, 100),    # 0 (Angry) to 1 (Happy)
    'escalation_score': []                              # The Target (Y) we will "simulate"
}

# Logic to make the data realistic: 
# Risk increases with age and bounces, decreases with high sentiment
for i in range(100):
    score = (data['age_days'][i] * 2) + (data['reassignment_count'][i] * 5) + ((1 - data['sentiment_score'][i]) * 20)
    data['escalation_score'].append(min(score, 100)) # Cap at 100

df = pd.DataFrame(data)

# --- 🚨 INJECTING DIRTY DATA FOR DEMO 🚨 ---
# Let's simulate that 10% of tickets have MISSING reassignment counts 
# (e.g., Imported from an old system that didn't track bounces)
df.loc[df.sample(frac=0.1).index, 'reassignment_count'] = np.nan

print("Sample of Raw Data (Notice the NaNs):")
print(df.tail(10))

### Step 2: Data Pre-work & Cleaning (The "Why")
**Why we do this:** ServiceNow exports often contain NaN (null) values if an agent forgot to fill a field. A Linear Regression model will **crash** if it sees a "Null" or "Text" value.

In [ ]:
# Check for missing values before cleaning
print(f"Missing values before cleaning:\n{df.isnull().sum()}\n")

# 1. Handling Missing Values (Imputation)
# We assume that if reassignment_count is missing, it likely bounced 0 times.
df['reassignment_count'] = df['reassignment_count'].fillna(0)

# 2. Feature Selection
# We drop 'incident_id' because it's a unique string. 
# It doesn't help the math; it's just a label.
features = df[['priority', 'reassignment_count', 'age_days', 'sentiment_score']]
target = df['escalation_score']

print("Cleaning complete. Missing values fixed.")
print(df.isnull().sum())

### Step 3: Exploratory Data Analysis (EDA)
**Why we do this:** We need to see if there is actually a "Linear" relationship. If the dots don't look like they could form a line, regression won't work.

In [ ]:
# Visualizing the relationship between Reassignments and Risk
plt.figure(figsize=(10, 6))
sns.scatterplot(x='reassignment_count', y='escalation_score', data=df)
plt.title('Impact of Bounces on Escalation Risk')
plt.xlabel('Reassignment Count')
plt.ylabel('Escalation Risk Score')
plt.show()

# Heatmap to see correlations
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

### Step 4: Split and Train the Model
**Why we do this:** We hide 20% of the data from the model. We train it on 80%, then "test" it on the 20% it hasn't seen to see if it can actually predict the future.

In [ ]:
# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Initialize and Train the Multiple Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model Training Complete.")

### Step 5: Interpretation (The Algebra) - What do the numbers mean?
**Why we do this:** To see which factor is the most "dangerous" for support.

**In Plain English:**
Think of a "Coefficient" as a **multiplier** or a **weight**:
1.  **Positive Coefficient (+):** If this goes UP, Risk goes UP. (e.g., More bounces = More Risk).
2.  **Negative Coefficient (-):** If this goes UP, Risk goes DOWN. (e.g., Higher Sentiment = Lower Risk).
3.  **The Size:** A coefficient of `5.0` is much more impactful than `0.5`. It means that factor is "heavier" in the risk calculation.

In [ ]:
# Look at the coefficients (the 'Weights')
coeff_df = pd.DataFrame(model.coef_, features.columns, columns=['Coefficient'])
print(coeff_df)

# Interpretation helper
print("\nInterpretation:")
print(f"For every 1 extra bounce (reassignment), the Risk Score jumps by {model.coef_[1]:.2f} points (approx).")

### Step 6: Model Evaluation - Different Scores Explained

Business stakeholders love **$R^2$**, but Data Scientists often look at error metrics to know exactly *how wrong* a model is on average.

**1. $R^2$ (R-Squared):** 
*   **What it is:** The "Grade" of the model (0 to 1). 
*   **Best for:** Explaining to the boss. "We explain 90% of the risk."

**2. MAE (Mean Absolute Error):**
*   **What it is:** The average "miss" distance. 
*   **Interpretation:** If MAE is 5.0, it means our predictions are usually off by about 5 points (plus or minus).

**3. MSE (Mean Squared Error):**
*   **What it is:** The average squared miss. 
*   **Interpretation:** Penalizes HUGE misses heavily. If the model makes one giant mistake, this number explodes. Useful for debugging bad outliers.

**4. RMSE (Root Mean Squared Error):**
*   **What it is:** The square root of MSE.
*   **Interpretation:** Similar to MAE but punishes large errors more. Often used as the standard "Accuracy" metric in competitions.

In [ ]:
# Calculate Predictions
y_pred = model.predict(X_test)

# Calculate Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Model R-Squared (The Grade): {r2:.4f}")
print(f"Mean Absolute Error (Avg Miss): {mae:.2f} points")
print(f"Root Mean Squared Error (Strict Miss): {rmse:.2f} points")

if r2 > 0.8:
    print("\n✅ Success: The model captures the majority of the risk factors.")
else:
    print("\n⚠️ Warning: The model may need more features to be accurate.")

## 3. How to use this in a ServiceNow Project

If you are presenting this to your team, here is the **"Translation Layer"**:

1.  **The Inputs (X):** These are your ServiceNow fields (`u_bounces`, `calendar_st_duration`).
2.  **The Target (Y):** This is a custom "Risk Score" field you add to the Incident form.
3.  **The Output:** You can use the **ServiceNow Predictive Intelligence API** to run this Python logic inside the platform. 

**Logic:** `When [Risk Score] hits > 80, the model triggers an Automated Escalation Notification to the Product Support Manager.`